<a href="https://colab.research.google.com/github/ankitsingh435517/lichess-gpt/blob/main/lichess_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [91]:
!sudo apt-get install zstd
!pip install chess

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [92]:
!wget -qO- https://database.lichess.org/standard/lichess_db_standard_rated_2026-08.pgn.zst | zstd -d | head -n 1000 > games.pgn

In [93]:
!wc -l games.pgn

1000 games.pgn


In [105]:
import chess.pgn

games = []
with open('games.pgn', 'r', encoding='utf-8') as f:
  while game := chess.pgn.read_game(f):
    moves = []
    board = game.board()

    for move in game.mainline_moves():
      moves.append(board.san(move))
      board.push(move)

    if moves:
      games.append("".join(moves))

In [106]:
len(games)

46

In [96]:
# build vocab
text = "".join(games)
chars = ['.'] + sorted(set(text))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: "".join(itos[i] for i in l)

print(encode("e4"))
print(decode(encode("e4")))

[23, 7]
e4


In [97]:
# tokenization - create a function which tokenizes into character level or BPE # train on both and check what is the different in training efficiency
# Character level tokenization

# compile dataset into train, eval and test split - 90, 5, 5 split
# the spit takes 90% of data to train due to the fact that the whole dataset is roughly 5000 games
# so 250 games to validate and test is sufficient enough and we need atleast 4000 games to train
import random
import torch

# shuffle the games to make the dataset represent overall structures of the games and avoid certain kinds of biased structures if there exists example similar openings in first few games etc
random.shuffle(games)

n1 = int(0.9*(len(games))) # 90%
n2 = int(0.95*len(games)) # 95%

context_size = 3

def compile_dataset(games):
  # compile the data into X and Y (X is the input sequence & Y is the corresponding output sequence)
  # we need context length of input for which output has to be stored e.g 3 for now so 3 moves input then we should expect 4th move as output
  X, Y = [], []
  for game in games:
    context = [0] * context_size # 0 0 0 -> meaning a '.' representing no move as in start of the game.
    for move in game + ".":
      ix = stoi[move]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]

  return X, Y

def split_dataset(split="train"):
  # split
  data = {
      'train': games[:n1],
      'dev': games[n1:n2],
      'test': games[n2:]
  }[split]
  # compile
  X, Y = compile_dataset(data)
  X = torch.tensor(X)
  Y = torch.tensor(Y)
  return X, Y

Xtr, Ytr = split_dataset("train")
Xdev, Ydev = split_dataset("dev")
Xte, Yte = split_dataset("test")
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)
print(Xte.shape, Yte.shape)

torch.Size([8732, 3]) torch.Size([8732])
torch.Size([306, 3]) torch.Size([306])
torch.Size([619, 3]) torch.Size([619])


In [98]:
# create Embedding for inputs (token & position embedding - what & where)
# We need a shape of (vocab_size, embd_dim) so that we can pluck out examples batch like B, T, C # Batch, Token, Channel e.g 32, 3, 10 meaning 32 examples where each example has 3 chars as integers and each have 10 dimension vector to represent their vector embedding
import torch.nn as nn
import torch.nn.functional as F

g = torch.Generator().manual_seed(1337)

embd_size = vocab_size # small reasonable embedding capacity for a small model
embd_table = nn.Embedding(vocab_size, embd_size)

batch_size = 32 # standard 32 size of batch
h_dim = 200 # how many neurons in hidden layer

ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
x = Xtr[ix] # (32, 3)
y = Ytr[ix] # (32,)

emb = embd_table(x) # expecting 32, 3, 16 # B, T, C
B, T, C = emb.shape
emb = emb.view(B, T * C) # (32, 3 * 16)

# this emb input can now be fed into nn
hidden_layer = nn.Linear(T * C, h_dim) # (B, T * C) @ (T * C, h_dim) => (B, h_dim)
logit_layer = nn.Linear(h_dim, vocab_size) # (B, h_dim) @ (h_dim, V) => (B, V)

# apply layers - MLP
hidden = torch.relu(hidden_layer(emb)) # (B, h_dim)
logits = logit_layer(hidden) # shape expected: (B, V)

# find the loss of this batch
loss = F.cross_entropy(logits, y) # aggregated scalalr loss likelihood over all batch examples
print(loss)

tensor(3.3620, grad_fn=<NllLossBackward0>)


In [99]:
# optimization
# all the parameters
lr = 3e-4
wd = 0.01
params = (
    list(embd_table.parameters())
    + list(hidden_layer.parameters())
    + list(logit_layer.parameters())
)
optimizer = torch.optim.Adam(params, lr=lr, weight_decay=wd)

In [100]:
params = (
    list(embd_table.parameters())
    + list(hidden_layer.parameters())
    + list(logit_layer.parameters())
)

optimizer = torch.optim.AdamW(
    params, lr=1e-3, weight_decay=0
)

# Record the initial weights
before = logit_layer.weight.detach().clone()

for step in range(10000):
    # batch sample
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    x = Xtr[ix] # (32, 3)
    y = Ytr[ix] # (32,)

    # forward pass
    emb = embd_table(x).reshape(x.shape[0], -1)
    hidden = torch.relu(hidden_layer(emb))
    logits = logit_layer(hidden)
    loss = F.cross_entropy(logits, y)

    # backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # print loss
    if step % 1000 == 0:
        print(loss.item())

after = logit_layer.weight.detach()
print("Weight change:", (after - before).norm().item())

3.3233816623687744
1.7307127714157104
1.6099116802215576
1.7056879997253418
1.5518684387207031
1.453778862953186
1.7539151906967163
1.7149548530578613
1.3042230606079102
1.520451307296753
Weight change: 14.726140975952148


In [101]:
# eval - run loss on dev split
# batch sample
ix = torch.randint(0, Xdev.shape[0], (batch_size,), generator=g)
xd = Xdev[ix] # (32, 3)
yd = Ydev[ix] # (32,)

# forward pass
emb = embd_table(xd)
emb = emb.reshape(emb.shape[0], -1)
logits = logit_layer(torch.relu(hidden_layer(emb)))
loss = F.cross_entropy(logits, yd)
print(loss.item())

1.5945543050765991


In [102]:
# Evaluate on the entire validation set
with torch.no_grad():
    emb = embd_table(Xdev)
    emb = emb.reshape(emb.shape[0], -1)

    hidden = torch.relu(hidden_layer(emb))
    logits = logit_layer(hidden)

    loss = F.cross_entropy(logits, Ydev)

print("Validation loss:", loss.item())

Validation loss: 2.104203462600708


In [103]:
# compare the loss on entire train and val split
@torch.no_grad()
def evaluate(X, Y):
    emb = embd_table(X)
    emb = emb.reshape(emb.shape[0], -1)
    hidden = torch.relu(hidden_layer(emb))
    logits = logit_layer(hidden)
    return F.cross_entropy(logits, Y).item()

print("Train loss:", evaluate(Xtr, Ytr))
print("Validation loss:", evaluate(Xdev, Ydev))

Train loss: 1.4977220296859741
Validation loss: 2.104203462600708


In [104]:
# inference - run generation on test split